In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")

NUM_BINS = (6, 6, 6, 6)

state_bounds = list(zip(env.observation_space.low, env.observation_space.high))
state_bounds[1] = [-3.0, 3.0]
state_bounds[3] = [-3.5, 3.5]

bins = [np.linspace(b[0], b[1], num + 1)[1:-1] for b, num in zip(state_bounds, NUM_BINS)]

def discretize_state(observation):
    return tuple(int(np.digitize(obs, b)) for obs, b in zip(observation, bins))

def heuristic_policy(observation):
    pole_angle = observation[2]
    return 1 if pole_angle >= 0 else 0

V = {}
returns = {}
returns_count = {}
num_episodes = 2000
gamma = 0.99

for episode in range(num_episodes):
    episode_history = []
    observation, info = env.reset()
    done = False

    while not done:
        state_discrete = discretize_state(observation)
        action = heuristic_policy(observation)

        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        episode_history.append((state_discrete, reward))
        observation = next_observation

    G = 0
    for state, reward in reversed(episode_history):
        G = reward + gamma * G

        if state not in returns:
            returns[state] = 0.0
            returns_count[state] = 0

        returns[state] += G
        returns_count[state] += 1
        V[state] = returns[state] / returns_count[state]

env.close()

print(f"Evaluated {len(V)} unique discrete states.")
print("\nSample state values V(s):")
for sample_state in list(V.keys())[:5]:
    print(f"State Matrix Bins {sample_state}: Value = {V[sample_state]:.3f}")


Evaluated 75 unique discrete states.

Sample state values V(s):
State Matrix Bins (3, 3, 1, 2): Value = 10.749
State Matrix Bins (3, 3, 1, 1): Value = 2.359
State Matrix Bins (3, 4, 2, 1): Value = 11.998
State Matrix Bins (2, 4, 3, 1): Value = 11.785
State Matrix Bins (2, 3, 3, 1): Value = 18.937
